In [1]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *
from eval_functions import *

In [2]:
main_directory = 'model_output'
models = find_folders_with_output(main_directory)
save = True
eval_path = f"metrics"
log_path = "model_stats"
log_filename =  "missing_words_logfile.txt"
os.makedirs(eval_path, exist_ok=True)
os.makedirs(log_path, exist_ok=True)

model_output\Llama-3-70B-Instruct_prompt_5_shot
model_output\Llama-3-8B-Instruct-desc-0_4_shot
model_output\Llama-3-8B-Instruct-Gradient-1048k_15_shot
model_output\Llama-3-8B-Instruct_0_shot
model_output\Llama-3-8B-Instruct_5_shot


### Model Output to nice JSON and Failure 

In [3]:
def process_files(model, save=save):
    input_dir = f"model_output/{model}/output/"
    output_dir = f"model_output/{model}/formatted/"
    failure_dir = f"model_output/{model}/failed/"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)
    files = os.listdir(input_dir)
    len_files = len(files)
    for filename in files:
        if filename.endswith('.json'):
            input_file_path = os.path.join(input_dir, filename)
            output_file_path = os.path.join(output_dir, filename)
            failure_file_path = os.path.join(failure_dir, filename)
            try:
                file = read_json(input_file_path)
                #print(f"Processing file: {filename}")
                if save:
                    shutil.copy(input_file_path, output_file_path)
                    #save_json_to_file(file, output_file_path)
            except Exception as e:
                #print(f"Error processing file {filename}: {e}")
                if save:
                    shutil.copy(input_file_path, failure_file_path)
    return len_files

In [4]:
for model in models:
    print(model)
    # 1 - Preprocess Files
    num_out_files = process_files(model, save=save)
    # 2 - Evaluate Files
    p2_label_path = "../../chia_label/p2"
    ready_path = f"model_output/{model}/ready"
    failed_model_path = f"model_output/{model}/failed_inner"
    p2_model_formatted_path = f"model_output/{model}/formatted"
    for path in [ready_path, failed_model_path, p2_model_formatted_path]:
        os.makedirs(path, exist_ok=True)
    
    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
    model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}
    
    common_ncts = set(label_files.keys()).intersection(model_files.keys())
    labels = []
    predictions = []
    success_data = []
    model_stats_path = os.path.join(log_path, f'{model}_stats.txt')
    missing_words_logfile = os.path.join(log_path, f'{model}_missing_words_logfile.txt')
    
    for nct in common_ncts:
        try:
            label_data = read_json(label_files[nct])
            model_data = read_json(model_files[nct])
            
            label_text = json_to_text(label_data)
            model_text = json_to_text(model_data)
        
            bleu_score = calculate_bleu(reference=label_text, hypothesis=model_text)
            jaccard_score = jaccard_similarity(label_text, model_text)
  
            print(f"BLEU Score: {bleu_score}")
            print(f"Jaccard Similarity: {jaccard_score}")
        
            label_structure = extract_logical_structure(label_data)
            model_structure = extract_logical_structure(model_data)
            # Versuche Wörter zu zählen
            label_raw_texts = extract_raw_texts(label_data)
            model_raw_texts = extract_raw_texts(model_data)
            label_words = extract_words(label_raw_texts)
            model_words = extract_words(model_raw_texts)
            missing_words = label_words - model_words
            missing_words_pct = len(missing_words) / len(label_words) * 100 if label_words else 0
    
            with open(missing_words_logfile, 'a', encoding='utf-8') as outfile:
                outfile.write(f"{nct};   Missing Words: {round(missing_words_pct, 2)} %   ;  Is Subset:  {label_words.issubset(model_words)}\n")
                outfile.write("Label Text \n")
                outfile.write(f"{label_words} \n")
                outfile.write("\nMissing Words \n")
                outfile.write(f"{missing_words}\n")
                outfile.write("\nModel Text \n")
                outfile.write(f"{model_words}\n")
                outfile.write("\n\n")
            with open(model_stats_path, 'a', encoding='utf-8') as outfile:
                if not label_words.issubset(model_words):
                    outfile.write(f"{nct} [{model}] Missing Words: {round(missing_words_pct, 2)} %   ({len(missing_words)}) \n ")
    
            
            success_data.append({
                'NCT': nct,
                'label_AND': label_structure.get('AND', 0),
                'label_OR': label_structure.get('OR', 0),
                'label_NOT': label_structure.get('NOT', 0),
                'label_DEPTH': label_structure.get('depth', 0),
                'model_AND': model_structure.get('AND', 0),
                'model_OR': model_structure.get('OR', 0),
                'model_NOT': model_structure.get('NOT', 0),
                'model_DEPTH': model_structure.get('depth', 0),
                'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
                'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
                'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
                'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0,
                'num_out_files': num_out_files,
                'bleu_score': bleu_score,
                'jaccard_score': jaccard_score
            })
            labels.append(label_structure)
            predictions.append(model_structure)
            save and shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
        except Exception as e:
            #print(f"Error processing NCT {nct}: {e}")
            save and shutil.move(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))
    
    df_success = pd.DataFrame(success_data).set_index('NCT')
    df_success.to_csv(eval_path+f'/{model}_eval.csv')
    
    
    true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
    predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values
    
    metrics = {}
    for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
        y_true = true_values[:, i]
        y_pred = predicted_values[:, i]
    
        diffs = y_pred - y_true
        pct_greater = (diffs > 0).sum() / len(diffs) * 100
        pct_less = (diffs < 0).sum() / len(diffs) * 100
        pct_equal = (diffs == 0).sum() / len(diffs) * 100
    
        metrics[metric] = {
            'accuracy': round(accuracy_score(y_true, y_pred), 3),
            'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'pct_greater': round(pct_greater, 2),
            'pct_less': round(pct_less, 2),
            'pct_equal': round(pct_equal, 2)
            # 'confusion_matrix': confusion_matrix(y_true, y_pred)
        }
        metrics_df = pd.DataFrame(metrics).T
        num_nct_files = len(df_success)
        metrics_df['num_nct_files'] = num_nct_files
        metrics_df['model_name'] = model
        metrics_df['num_files'] = num_out_files
        metrics_df['mean_jacard'] = df_success['jaccard_score'].mean()
        metrics_df['mean_bleu'] = df_success['bleu_score'].mean()
        # Save Metrics to CSV
        metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))    

Llama-3-70B-Instruct_prompt_5_shot
BLEU Score: 0.20372185846902108
Jaccard Similarity: 0.9333333333333333
BLEU Score: 0.8357834621458422
Jaccard Similarity: 1.0
BLEU Score: 0.6004709232471797
Jaccard Similarity: 0.9722222222222222
BLEU Score: 1.0
Jaccard Similarity: 1.0
BLEU Score: 0.6861235831719544
Jaccard Similarity: 0.9682539682539683
BLEU Score: 0.1453633363766496
Jaccard Similarity: 0.6296296296296297
BLEU Score: 0.7839645629398492
Jaccard Similarity: 0.9696969696969697
BLEU Score: 0.5137322245932251
Jaccard Similarity: 0.875
BLEU Score: 0.6744322250214191
Jaccard Similarity: 0.84375
BLEU Score: 0.8003203203844999
Jaccard Similarity: 0.8387096774193549
BLEU Score: 0.7925656562480038
Jaccard Similarity: 0.9545454545454546
BLEU Score: 0.8021480092195581
Jaccard Similarity: 1.0
BLEU Score: 0.6000149146751644
Jaccard Similarity: 0.9411764705882353
BLEU Score: 0.20630596254262776
Jaccard Similarity: 1.0
BLEU Score: 1.0
Jaccard Similarity: 1.0
BLEU Score: 1.0
Jaccard Similarity: 1.0
BL

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

BLEU Score: 0.8066926455054959
Jaccard Similarity: 0.9761904761904762
BLEU Score: 1.0
Jaccard Similarity: 1.0
BLEU Score: 0.7728642943246548
Jaccard Similarity: 0.9807692307692307
BLEU Score: 0.6967313004719417
Jaccard Similarity: 0.9722222222222222
BLEU Score: 0.7345336948322937
Jaccard Similarity: 0.9444444444444444
BLEU Score: 0.38043020300041663
Jaccard Similarity: 0.9464285714285714
BLEU Score: 0.21796140327686664
Jaccard Similarity: 0.7692307692307693
BLEU Score: 0.5849183137732418
Jaccard Similarity: 1.0
BLEU Score: 0.6369611890981433
Jaccard Similarity: 0.9838709677419355
BLEU Score: 0.6680281311375104
Jaccard Similarity: 0.9365079365079365
BLEU Score: 0.7397568039752616
Jaccard Similarity: 0.967741935483871
BLEU Score: 0.6136185774686049
Jaccard Similarity: 1.0
BLEU Score: 0.9278982724420874
Jaccard Similarity: 0.8484848484848485
BLEU Score: 0.6643108731069387
Jaccard Similarity: 1.0
BLEU Score: 0.7354200975172586
Jaccard Similarity: 0.9821428571428571
BLEU Score: 0.7102545682

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


BLEU Score: 0.039320805071185176
Jaccard Similarity: 0.6153846153846154
BLEU Score: 0.056542357006436936
Jaccard Similarity: 0.7647058823529411
BLEU Score: 0.536550941954541
Jaccard Similarity: 0.975609756097561
BLEU Score: 0.024499250204683184
Jaccard Similarity: 0.6818181818181818
BLEU Score: 0.4797543511401896
Jaccard Similarity: 0.8148148148148148
BLEU Score: 0.41514123471842546
Jaccard Similarity: 0.9166666666666666
BLEU Score: 0.24603056804449855
Jaccard Similarity: 0.7555555555555555
BLEU Score: 1.0
Jaccard Similarity: 1.0
BLEU Score: 1.821831989445342e-231
Jaccard Similarity: 1.0
BLEU Score: 0.532565224962424
Jaccard Similarity: 1.0
BLEU Score: 0.04072882368109567
Jaccard Similarity: 0.7213114754098361
BLEU Score: 0.21776342142145325
Jaccard Similarity: 0.8571428571428571
BLEU Score: 0.5953979204763056
Jaccard Similarity: 0.9523809523809523
BLEU Score: 0.07552653047520115
Jaccard Similarity: 0.813953488372093
BLEU Score: 0.39073802494525
Jaccard Similarity: 0.8636363636363636
B

...

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

BLEU Score: 0.19737303843980533
Jaccard Similarity: 0.95
BLEU Score: 0.891237063632542
Jaccard Similarity: 0.9722222222222222
BLEU Score: 0.7456495553738954
Jaccard Similarity: 0.9705882352941176
BLEU Score: 2.122130208047169e-05
Jaccard Similarity: 0.4603174603174603
BLEU Score: 0.5022196301414892
Jaccard Similarity: 0.8703703703703703
BLEU Score: 0.4440769645166335
Jaccard Similarity: 0.9411764705882353
BLEU Score: 0.14768608153738072
Jaccard Similarity: 0.7708333333333334
BLEU Score: 0.3771701894792374
Jaccard Similarity: 0.7058823529411765
BLEU Score: 1.0
Jaccard Similarity: 1.0
BLEU Score: 0.46714747111853244
Jaccard Similarity: 0.9523809523809523
BLEU Score: 0.4974735270563152
Jaccard Similarity: 0.9142857142857143
BLEU Score: 0.40043536684050496
Jaccard Similarity: 0.9354838709677419
BLEU Score: 0.7611606003349892
Jaccard Similarity: 0.8571428571428571
BLEU Score: 3.434046030282548e-78
Jaccard Similarity: 0.8
BLEU Score: 0.6129752413741056
Jaccard Similarity: 0.9473684210526315


C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

BLEU Score: 0.5635468592374524
Jaccard Similarity: 1.0
BLEU Score: 0.6739047062564734
Jaccard Similarity: 0.8148148148148148
BLEU Score: 0.6709045062449255
Jaccard Similarity: 1.0
BLEU Score: 0.5741982558870883
Jaccard Similarity: 0.8809523809523809
BLEU Score: 0.4366835442847812
Jaccard Similarity: 0.8461538461538461
BLEU Score: 0.003918449090258843
Jaccard Similarity: 0.7
BLEU Score: 0.10300668489203793
Jaccard Similarity: 0.7
BLEU Score: 0.001071205802613496
Jaccard Similarity: 0.62
BLEU Score: 6.921464794211712e-232
Jaccard Similarity: 0.075
BLEU Score: 2.3852040572439896e-09
Jaccard Similarity: 0.4032258064516129
BLEU Score: 0.27807317355422395
Jaccard Similarity: 0.7058823529411765
BLEU Score: 0.4621980554275651
Jaccard Similarity: 0.851063829787234
BLEU Score: 0.020408777292195916
Jaccard Similarity: 0.4411764705882353
BLEU Score: 0.5139046501670766
Jaccard Similarity: 0.9772727272727273
BLEU Score: 0.6548659374011372
Jaccard Similarity: 0.9387755102040817
BLEU Score: 0.33717174

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [10]:
metrics_df# NCT02935855_inc